In [2]:
#unzip ETA dataset
!unzip "/content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip" -d "/content/drive/MyDrive/Capstone/Datasets/ETA/"

unzip:  cannot find or open /content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip, /content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip.zip or /content/drive/MyDrive/Capstone/Datasets/mta_1706.csv.zip.ZIP.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
# path of the unzipped dataset
path_csv = "/Users/florenciagonzalez/Documents/bus-prediction-system/ml/eta/data/mta_1706.csv"

print("Cargando dataset...")
#skips the corrupted lines of the dataset to prevent crashing
df = pd.read_csv(path_csv, on_bad_lines='skip', low_memory=False)
print("¡Dataset cargado exitosamente!")
#prints
print(df.head())

Cargando dataset...
¡Dataset cargado exitosamente!
        RecordedAtTime  DirectionRef PublishedLineName  \
0  2017-06-01 00:03:34             0                B8   
1  2017-06-01 00:03:43             1               S61   
2  2017-06-01 00:03:49             0              Bx10   
3  2017-06-01 00:03:31             0                Q5   
4  2017-06-01 00:03:22             1               Bx1   

                  OriginName  OriginLat  OriginLong  \
0                 4 AV/95 ST  40.616104  -74.031143   
1  ST GEORGE FERRY/S61 & S91  40.643169  -74.073494   
2     E 206 ST/BAINBRIDGE AV  40.875008  -73.880142   
3           TEARDROP/LAYOVER  40.701748  -73.802399   
4      RIVERDALE AV/W 231 ST  40.881187  -73.909340   

                   DestinationName  DestinationLat  DestinationLong  \
0          BROWNSVILLE ROCKAWAY AV       40.656048       -73.907379   
1                S I MALL YUKON AV       40.575935       -74.167686   
2                 RIVERDALE 263 ST       40.912376      

In [3]:
print("Tomando muestra de 3,000,000 filas...")
#takes a subset of 3M from the dataset
df_sample = df.sample(n=3000000, random_state=42)
print("Muestra tomada.")
df_sample.head()

Tomando muestra de 3,000,000 filas...
Muestra tomada.


,RecordedAtTime,DirectionRef,PublishedLineName,OriginName,OriginLat,OriginLong,DestinationName,DestinationLat,DestinationLong,VehicleRef,VehicleLocation.Latitude,VehicleLocation.Longitude,NextStopPointName,ArrivalProximityText,DistanceFromStop,ExpectedArrivalTime,ScheduledArrivalTime
2170987,2017-06-10 16:09:29,0,B35,CHURCH AV/MC DONALD AV,40.642651,-73.979584,BROWNSVILLE M GASTON BL via CHURCH,40.656345,-73.907188,NYCT_327,40.645069,-73.974039,CHURCH AV/OCEAN PY,at stop,2.0,2017-06-10 16:09:48,16:08:17
5287764,2017-06-24 17:02:27,1,M15-SBS,E 126 ST/2 AV,40.803230,-73.932449,SELECT BUS SERVICE SOUTH FERRY via 2 AV,40.702122,-74.013664,NYCT_1268,40.740819,-73.978765,2 AV/E 23 ST,< 1 stop away,420.0,2017-06-24 17:05:00,16:47:57
4304178,2017-06-20 14:45:31,0,Bx3,BROADWAY/179 ST,40.849327,-73.936508,RIVERDALE BWAY - 238 ST,40.885086,-73.900436,NYCT_4372,40.856785,-73.909881,UNIVERSITY AV/W 181 ST,approaching,79.0,2017-06-20 14:46:03,14:33:37
1255600,2017-06-06 17:42:56,1,B52,PALMETTO ST/ST NICHOLAS AV,40.700420,-73.909996,DNTWN BKLYN TILLARY ST via GATES,40.693939,-73.990311,NYCT_4528,40.700726,-73.910135,GATES AV/WYCKOFF AV,< 1 stop away,202.0,NaN,17:49:48
1135968,2017-06-06 09:22:54,0,Q88,92 ST/59 AV,40.734272,-73.869301,QUEENS VILL JAMAICA AV,40.717922,-73.736382,NYCT_6889,40.740524,-73.786886,188 ST/HORACE HARDING EXP,approaching,58.0,2017-06-06 09:23:18,08:54:35


In [4]:
# --- 1.CLEANING ROWS WITH NULL VALUES ---
print("Eliminando filas con valores nulos en columnas clave...")
# removes rows that dont have these 3 columns
#model can't learn without them
df_processed = df_sample.dropna(subset=['ExpectedArrivalTime', 'RecordedAtTime', 'DistanceFromStop']).copy()
print(f"Filas después de limpiar nulos: {len(df_processed)}")
df_processed.head()

Eliminando filas con valores nulos en columnas clave...
Filas después de limpiar nulos: 2610484


,RecordedAtTime,DirectionRef,PublishedLineName,OriginName,OriginLat,OriginLong,DestinationName,DestinationLat,DestinationLong,VehicleRef,VehicleLocation.Latitude,VehicleLocation.Longitude,NextStopPointName,ArrivalProximityText,DistanceFromStop,ExpectedArrivalTime,ScheduledArrivalTime
2170987,2017-06-10 16:09:29,0,B35,CHURCH AV/MC DONALD AV,40.642651,-73.979584,BROWNSVILLE M GASTON BL via CHURCH,40.656345,-73.907188,NYCT_327,40.645069,-73.974039,CHURCH AV/OCEAN PY,at stop,2.0,2017-06-10 16:09:48,16:08:17
5287764,2017-06-24 17:02:27,1,M15-SBS,E 126 ST/2 AV,40.803230,-73.932449,SELECT BUS SERVICE SOUTH FERRY via 2 AV,40.702122,-74.013664,NYCT_1268,40.740819,-73.978765,2 AV/E 23 ST,< 1 stop away,420.0,2017-06-24 17:05:00,16:47:57
4304178,2017-06-20 14:45:31,0,Bx3,BROADWAY/179 ST,40.849327,-73.936508,RIVERDALE BWAY - 238 ST,40.885086,-73.900436,NYCT_4372,40.856785,-73.909881,UNIVERSITY AV/W 181 ST,approaching,79.0,2017-06-20 14:46:03,14:33:37
1135968,2017-06-06 09:22:54,0,Q88,92 ST/59 AV,40.734272,-73.869301,QUEENS VILL JAMAICA AV,40.717922,-73.736382,NYCT_6889,40.740524,-73.786886,188 ST/HORACE HARDING EXP,approaching,58.0,2017-06-06 09:23:18,08:54:35
1282111,2017-06-06 19:22:57,0,M7,AV OF THE AMERICAS/W 14 ST,40.737930,-73.996346,HARLEM 147 ST via 6 AV via AMSTERDAM,40.821110,-73.935898,NYCT_6759,40.781203,-73.979741,AMSTERDAM AV/W 77 ST,approaching,113.0,2017-06-06 19:23:40,19:02:37


In [5]:
# --- 2. Convertir columnas de tiempo ---
print("Convirtiendo columnas de tiempo...")
df_processed['ExpectedArrivalTime'] = pd.to_datetime(df_processed['ExpectedArrivalTime'])
df_processed['RecordedAtTime'] = pd.to_datetime(df_processed['RecordedAtTime'])

Convirtiendo columnas de tiempo...


In [6]:
#imprimir para ver como quedaron las fechas/horas de las columnas
print(df_processed.head())

             RecordedAtTime  DirectionRef PublishedLineName  \
2170987 2017-06-10 16:09:29             0               B35   
5287764 2017-06-24 17:02:27             1           M15-SBS   
4304178 2017-06-20 14:45:31             0               Bx3   
1135968 2017-06-06 09:22:54             0               Q88   
1282111 2017-06-06 19:22:57             0                M7   

                         OriginName  OriginLat  OriginLong  \
2170987      CHURCH AV/MC DONALD AV  40.642651  -73.979584   
5287764               E 126 ST/2 AV  40.803230  -73.932449   
4304178             BROADWAY/179 ST  40.849327  -73.936508   
1135968                 92 ST/59 AV  40.734272  -73.869301   
1282111  AV OF THE AMERICAS/W 14 ST  40.737930  -73.996346   

                                 DestinationName  DestinationLat  \
2170987       BROWNSVILLE M GASTON BL via CHURCH       40.656345   
5287764  SELECT BUS SERVICE SOUTH FERRY via 2 AV       40.702122   
4304178                  RIVERDALE BWAY - 23

In [7]:
# --- 3. CREATES TARGET (y) 'time_diff' ---
# This is the value we want to predict ETA
print("Creando el target (y) 'TimeToArrival' (en minutos)...")
#target prediction = expected arrival time - the time the bus was recorded
df_processed['TimeToArrival'] = (df_processed['ExpectedArrivalTime'] - df_processed['RecordedAtTime']).dt.total_seconds()/60
df_processed.head()

Creando el target (y) 'TimeToArrival' (en minutos)...


,RecordedAtTime,DirectionRef,PublishedLineName,OriginName,OriginLat,OriginLong,DestinationName,DestinationLat,DestinationLong,VehicleRef,VehicleLocation.Latitude,VehicleLocation.Longitude,NextStopPointName,ArrivalProximityText,DistanceFromStop,ExpectedArrivalTime,ScheduledArrivalTime,TimeToArrival
2170987,2017-06-10 16:09:29,0,B35,CHURCH AV/MC DONALD AV,40.642651,-73.979584,BROWNSVILLE M GASTON BL via CHURCH,40.656345,-73.907188,NYCT_327,40.645069,-73.974039,CHURCH AV/OCEAN PY,at stop,2.0,2017-06-10 16:09:48,16:08:17,0.316667
5287764,2017-06-24 17:02:27,1,M15-SBS,E 126 ST/2 AV,40.803230,-73.932449,SELECT BUS SERVICE SOUTH FERRY via 2 AV,40.702122,-74.013664,NYCT_1268,40.740819,-73.978765,2 AV/E 23 ST,< 1 stop away,420.0,2017-06-24 17:05:00,16:47:57,2.550000
4304178,2017-06-20 14:45:31,0,Bx3,BROADWAY/179 ST,40.849327,-73.936508,RIVERDALE BWAY - 238 ST,40.885086,-73.900436,NYCT_4372,40.856785,-73.909881,UNIVERSITY AV/W 181 ST,approaching,79.0,2017-06-20 14:46:03,14:33:37,0.533333
1135968,2017-06-06 09:22:54,0,Q88,92 ST/59 AV,40.734272,-73.869301,QUEENS VILL JAMAICA AV,40.717922,-73.736382,NYCT_6889,40.740524,-73.786886,188 ST/HORACE HARDING EXP,approaching,58.0,2017-06-06 09:23:18,08:54:35,0.400000
1282111,2017-06-06 19:22:57,0,M7,AV OF THE AMERICAS/W 14 ST,40.737930,-73.996346,HARLEM 147 ST via 6 AV via AMSTERDAM,40.821110,-73.935898,NYCT_6759,40.781203,-73.979741,AMSTERDAM AV/W 77 ST,approaching,113.0,2017-06-06 19:23:40,19:02:37,0.716667


In [8]:
# --- 4. CLEANS THE TARGET
print("Limpiando el target...")

rows_before = len(df_processed)
#RULE 1: deletes the rows where the bus already passed the stop (ETA>0)
df_processed = df_processed[df_processed['TimeToArrival'] > 0]
print(f"Filas eliminadas (tiempo negativo): {rows_before - len(df_processed)}")

Limpiando el target...
Filas eliminadas (tiempo negativo): 1347


In [9]:
# Regla 2: Eliminar outliers (donde el bus está a más de 1 hora )
# El notebook asume que cualquier predicción mayor a 1 hora es un error o un outlier
rows_before = len(df_processed)
#predictions > 1h are unreliable for city transit and might represent
#buses that are off-duty --> DELETE THEM
df_processed = df_processed[df_processed['TimeToArrival'] < 60]
print(f"Filas eliminadas (outliers > 1 hora): {rows_before - len(df_processed)}")

Filas eliminadas (outliers > 1 hora): 0


In [10]:
# --- 5. FEATURE ENGINERRING ---
# El notebook extrae todas las partes de la fecha
print("Creando features de tiempo (X)...")
#decompose them into numerical features so the model can learn patterns
df_processed['month'] = df_processed['RecordedAtTime'].dt.month
df_processed['day'] = df_processed['RecordedAtTime'].dt.day
df_processed['day_of_week'] = df_processed['RecordedAtTime'].dt.dayofweek # Lunes=0, Domingo=6
df_processed['hour'] = df_processed['RecordedAtTime'].dt.hour
df_processed['minute'] = df_processed['RecordedAtTime'].dt.minute

df_processed.head()

Creando features de tiempo (X)...


,RecordedAtTime,DirectionRef,PublishedLineName,OriginName,OriginLat,OriginLong,DestinationName,DestinationLat,DestinationLong,VehicleRef,...,ArrivalProximityText,DistanceFromStop,ExpectedArrivalTime,ScheduledArrivalTime,TimeToArrival,month,day,day_of_week,hour,minute
2170987,2017-06-10 16:09:29,0,B35,CHURCH AV/MC DONALD AV,40.642651,-73.979584,BROWNSVILLE M GASTON BL via CHURCH,40.656345,-73.907188,NYCT_327,...,at stop,2.0,2017-06-10 16:09:48,16:08:17,0.316667,6,10,5,16,9
5287764,2017-06-24 17:02:27,1,M15-SBS,E 126 ST/2 AV,40.803230,-73.932449,SELECT BUS SERVICE SOUTH FERRY via 2 AV,40.702122,-74.013664,NYCT_1268,...,< 1 stop away,420.0,2017-06-24 17:05:00,16:47:57,2.550000,6,24,5,17,2
4304178,2017-06-20 14:45:31,0,Bx3,BROADWAY/179 ST,40.849327,-73.936508,RIVERDALE BWAY - 238 ST,40.885086,-73.900436,NYCT_4372,...,approaching,79.0,2017-06-20 14:46:03,14:33:37,0.533333,6,20,1,14,45
1135968,2017-06-06 09:22:54,0,Q88,92 ST/59 AV,40.734272,-73.869301,QUEENS VILL JAMAICA AV,40.717922,-73.736382,NYCT_6889,...,approaching,58.0,2017-06-06 09:23:18,08:54:35,0.400000,6,6,1,9,22
1282111,2017-06-06 19:22:57,0,M7,AV OF THE AMERICAS/W 14 ST,40.737930,-73.996346,HARLEM 147 ST via 6 AV via AMSTERDAM,40.821110,-73.935898,NYCT_6759,...,approaching,113.0,2017-06-06 19:23:40,19:02:37,0.716667,6,6,1,19,22


In [11]:
# --- 6. SELECT FINAL COLUMNS ---
print("Seleccionando columnas finales para el modelo...")
# Estas son las columnas que el notebook identifica como las features
#input
features = ['DistanceFromStop', 'month', 'day', 'day_of_week', 'hour', 'minute']
target = 'TimeToArrival' #value to predict = output

Seleccionando columnas finales para el modelo...


In [12]:
# Creamos el DataFrame final listo para el entrenamiento
df_final = df_processed[features + [target]]

In [13]:
# --- 7. Verificación Final ---
print("\n--- ¡Preprocesamiento completado! ---")
print(f"Total de filas listas para entrenar: {len(df_final)}")
print(df_final.head())


--- ¡Preprocesamiento completado! ---
Total de filas listas para entrenar: 2609137
         DistanceFromStop  month  day  day_of_week  hour  minute  \
2170987               2.0      6   10            5    16       9   
5287764             420.0      6   24            5    17       2   
4304178              79.0      6   20            1    14      45   
1135968              58.0      6    6            1     9      22   
1282111             113.0      6    6            1    19      22   

         TimeToArrival  
2170987       0.316667  
5287764       2.550000  
4304178       0.533333  
1135968       0.400000  
1282111       0.716667  


In [14]:
#SAVE THE CLEANED AND PROCESSED DATA INTO A CSV
print("Guardando el DataFrame preprocesado en un nuevo CSV...")

# Define la ruta de salida en tu Google Drive
ruta_salida = "/Users/florenciagonzalez/Documents/bus-prediction-system/ml/eta/data/mta_datos_limpios.csv"

# index=False es MUY importante para evitar que se guarde una columna extra
df_final.to_csv(ruta_salida, index=False)

print(f"¡Datos limpios guardados exitosamente en: {ruta_salida}")

Guardando el DataFrame preprocesado en un nuevo CSV...
¡Datos limpios guardados exitosamente en: /Users/florenciagonzalez/Documents/bus-prediction-system/ml/eta/data/mta_datos_limpios.csv


In [16]:
# 2. Cargar directamente los datos LIMPIOS
print("Cargando datos preprocesados...")
ruta_limpia = "/Users/florenciagonzalez/Documents/bus-prediction-system/ml/eta/data/mta_datos_limpios.csv"
df_final = pd.read_csv(ruta_limpia)

print("¡Datos limpios listos para entrenar!")
print(df_final.head())


Cargando datos preprocesados...
¡Datos limpios listos para entrenar!
   DistanceFromStop  month  day  day_of_week  hour  minute  TimeToArrival
0               2.0      6   10            5    16       9       0.316667
1             420.0      6   24            5    17       2       2.550000
2              79.0      6   20            1    14      45       0.533333
3              58.0      6    6            1     9      22       0.400000
4             113.0      6    6            1    19      22       0.716667


In [17]:
#PRINT TO CONFIRM THERE ARE NO NULL VALUES
print("\nValores nulos restantes (deberían ser 0):")
print(df_final.isnull().sum())
print("size of the clean dataset: ", df_final.size)


Valores nulos restantes (deberían ser 0):
DistanceFromStop    0
month               0
day                 0
day_of_week         0
hour                0
minute              0
TimeToArrival       0
dtype: int64
size of the clean dataset:  18263959


In [18]:
#TRAINING
#Imports the Random Forest algorithm, statistical metrics
#(MAE, MSE, R²), and joblib for saving the trained model.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [19]:
# Definir X e y
X = df_final[features] #inputs = 7 features
y = df_final[target] #output = target
print(X.size)
print(y.size)

15654822
2609137


In [20]:
# SPLIT to train
#80% to train and 20% to test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.size)
print(X_test.size)
print(y_train.size)
print(y_test.size)

12523854
3130968
2087309
521828


In [21]:
# TRAIN THE MODEL
#100 decision trees
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=15)
print("\nIniciando entrenamiento...")
#learning phase= where the model finds relationships between x and y
rf_model.fit(X_train, y_train)
print("¡Modelo entrenado!")


Iniciando entrenamiento...
¡Modelo entrenado!


In [22]:
# --- TESTING ---
# evaluate the model on unseen data (X_test)
y_pred = rf_model.predict(X_test)

# --- 2. Imprimir Comparación (solo las primeras 10) ---
# Imprimir todo el bucle (range(len(y_pred))) puede colapsar tu navegador.
# Usamos .iloc[i] para acceder a y_test, ya que conserva su índice original.
print("--- Comparación de Predicción vs. Real (primeras 10) ---")
for i in range(10):
    print(f"Predicción: {y_pred[i]:.2f}   |  Valor Real: {y_test.iloc[i]:.2f} ")

# --- METRICS  ---
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)



--- Comparación de Predicción vs. Real (primeras 10) ---
Predicción: 0.40   |  Valor Real: 0.53 
Predicción: 0.27   |  Valor Real: 0.55 
Predicción: 0.32   |  Valor Real: 0.42 
Predicción: 1.06   |  Valor Real: 0.92 
Predicción: 1.29   |  Valor Real: 1.93 
Predicción: 0.79   |  Valor Real: 0.70 
Predicción: 0.27   |  Valor Real: 0.20 
Predicción: 0.38   |  Valor Real: 0.43 
Predicción: 0.82   |  Valor Real: 2.12 
Predicción: 0.30   |  Valor Real: 0.12 


In [23]:
# --- 4. Imprimir Métricas ---
print("\n--- Métricas de Evaluación del Modelo ---")
print(f'Error Absoluto Medio (MAE):   {mae:.2f} minutos')
print(f'Error Cuadrático Medio (MSE): {mse:.2f}')
print(f'Raíz del ECM (RMSE):          {rmse:.2f} minutos')
print(f'Coeficiente R-cuadrado (R²):  {r2:.2f}')




--- Métricas de Evaluación del Modelo ---
Error Absoluto Medio (MAE):   0.35 minutos
Error Cuadrático Medio (MSE): 0.43
Raíz del ECM (RMSE):          0.66 minutos
Coeficiente R-cuadrado (R²):  0.84


In [24]:
# --- 5. Guardar el Modelo (con Joblib) ---
# Usamos joblib para modelos de Scikit-learn
model_filename = '/Users/florenciagonzalez/Documents/bus-prediction-system/ml/eta/model/rf_model_eta.pkl'
joblib.dump(rf_model, model_filename)

print(f"\n¡Modelo guardado exitosamente como '{model_filename}'!")


¡Modelo guardado exitosamente como '/Users/florenciagonzalez/Documents/bus-prediction-system/ml/eta/model/rf_model_eta.pkl'!
